In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"          # same value as your other notebooks
BASE   = f"{CATALOG}.silver.customer_base"
EVENTS = f"{CATALOG}.bronze.customer_cdc_events"
DIM    = f"{CATALOG}.gold.dim_customer_scd2"
QUAR   = f"{CATALOG}.silver.customer_cdc_quarantine"
LOG    = f"{CATALOG}.silver.cdc_batch_log"
STAGE  = f"{CATALOG}.silver.cdc_stage"

RESET = True    # True = rebuild everything from scratch (safe to rerun)
if RESET:
    for t in (BASE, EVENTS, DIM, QUAR, LOG, STAGE):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
print("ready")

ready


In [0]:
# --- base customer list from your silver orders, saved once so it stays stable ---
o = spark.table(f"{CATALOG}.silver.orders")
base_df = (o.groupBy("customer_id")
    .agg(F.expr("max_by(customer_name, order_date)").alias("customer_name"),
         F.expr("max_by(city, order_date)").alias("city"))
    .withColumn("loyalty_tier",
        F.element_at(F.array(F.lit("Basic"), F.lit("Plus"), F.lit("Premium")),
                     (F.pmod(F.hash("customer_id"), F.lit(3)) + 1).cast("int"))))
base_df.write.format("delta").mode("overwrite").saveAsTable(BASE)
base = spark.table(BASE)
print("customers:", base.count())

T1, T2, T3, T_LATE = ("2026-09-01 08:00:00", "2026-10-10 09:00:00",
                      "2026-10-20 09:00:00", "2026-08-31 09:00:00")
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Chennai", "Kolkata"]

def new_city(salt):
    idx = (F.pmod(F.hash(F.col("customer_id"), F.lit(salt)), F.lit(5)) + 1).cast("int")
    return F.element_at(F.array(*[F.lit(c) for c in CITIES]), idx)

def ev(df, batch_id, ts, op, tag):
    return df.select(
        F.concat(F.lit(f"B{batch_id}-{tag}-"), F.col("customer_id").cast("string")).alias("event_id"),
        F.lit(batch_id).alias("batch_id"),
        "customer_id",
        F.lit(op).alias("op"),
        "customer_name", "city", "loyalty_tier",
        F.to_timestamp(F.lit(ts)).alias("event_ts"))

h = F.pmod(F.hash("customer_id"), F.lit(10))

# Batch 1: initial load, one INSERT per customer
b1 = ev(base, 1, T1, "I", "ins")

# Batch 2: moves (SCD2), name corrections (SCD1), tier upgrades (SCD2), deletes
moved    = base.filter(h == 0).withColumn("city", new_city("b2"))
renamed  = base.filter(h == 1).withColumn("customer_name", F.concat("customer_name", F.lit(" Jr")))
upgraded = base.filter(h == 2).withColumn("loyalty_tier", F.lit("Premium"))
deleted  = base.filter(F.pmod(F.hash("customer_id"), F.lit(50)) == 3)

b2 = (ev(moved, 2, T2, "U", "mv")
      .unionByName(ev(renamed, 2, T2, "U", "nm"))
      .unionByName(ev(upgraded, 2, T2, "U", "up"))
      .unionByName(ev(deleted, 2, T2, "D", "del")))
dups = b2.limit(15)                      # the same events delivered twice
bad = spark.createDataFrame(
    [("B2-bad-1", 2, None, "U", "No Id", "Mumbai", "Basic", "2026-10-10 09:00:00"),
     ("B2-bad-2", 2, 1001, "X", "Bad Op", "Mumbai", "Basic", "2026-10-10 09:00:00"),
     ("B2-bad-3", 2, 1002, "U", "No City", None, "Basic", "2026-10-10 09:00:00")],
    "event_id string, batch_id int, customer_id int, op string, customer_name string, "
    "city string, loyalty_tier string, event_ts_str string"
).withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")
b2 = b2.unionByName(dups).unionByName(bad)

# Batch 3: more moves, tier changes, re-activated customers, and late (stale) events
moved3  = base.filter(h == 0).withColumn("city", new_city("b3"))
tier3   = base.filter(h == 5).withColumn("loyalty_tier", F.lit("Plus"))
late    = base.filter(h == 4).limit(5).withColumn("city", F.lit("Chennai"))
b3 = (ev(moved3, 3, T3, "U", "mv")
      .unionByName(ev(tier3, 3, T3, "U", "tr"))
      .unionByName(ev(deleted.limit(3), 3, T3, "I", "re"))
      .unionByName(ev(late, 3, T_LATE, "U", "late")))

events = b1.unionByName(b2).unionByName(b3)
(events.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(EVENTS))
display(spark.table(EVENTS).groupBy("batch_id", "op").count().orderBy("batch_id", "op"))

customers: 501


batch_id,op,count
1,I,501
2,D,3
2,U,161
2,X,1
3,I,3
3,U,92


In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {DIM} (
  customer_sk BIGINT, customer_id INT, customer_name STRING, city STRING, loyalty_tier STRING,
  valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN, is_deleted BOOLEAN
) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {LOG} (
  batch_id INT, raw_events INT, rejected_invalid INT, duplicates_removed INT,
  superseded_in_batch INT, inserted INT, new_versions INT, closed INT,
  no_change INT, quarantined_other INT, processed_at TIMESTAMP
) USING DELTA""")
print("tables ready")

tables ready


In [0]:
EV_COLS = ["event_id", "batch_id", "customer_id", "op", "customer_name",
           "city", "loyalty_tier", "event_ts"]

def apply_batch(batch_id):
    # idempotency: never apply the same batch twice
    if spark.table(LOG).filter(F.col("batch_id") == batch_id).count() > 0:
        print(f"batch {batch_id}: already processed, skipping")
        return

    raw = spark.table(EVENTS).filter(F.col("batch_id") == batch_id)
    n_raw = raw.count()

    # 1) validate: bad events are kept with a reason, not silently dropped
    checked = raw.withColumn("reject_reason",
        F.when(F.col("op").isNull() | ~F.col("op").isin("I", "U", "D"), "invalid op")
         .when(F.col("customer_id").isNull(), "missing customer_id")
         .when(F.col("event_ts").isNull(), "missing event_ts")
         .when(F.col("op").isin("I", "U") &
               (F.col("customer_name").isNull() | F.col("city").isNull() | F.col("loyalty_tier").isNull()),
               "missing attributes"))
    bad = checked.filter("reject_reason IS NOT NULL").select(*EV_COLS, "reject_reason")
    ok = checked.filter("reject_reason IS NULL").select(*EV_COLS)
    n_bad = bad.count()

    # 2) deduplicate events by event_id
    deduped = ok.dropDuplicates(["event_id"])
    n_dupes = ok.count() - deduped.count()

    # 3) keep only the latest event per customer in this batch
    w = Window.partitionBy("customer_id").orderBy(F.col("event_ts").desc(), F.col("event_id").desc())
    latest = deduped.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
    n_superseded = deduped.count() - latest.count()

    # 4) compare each event with the customer's CURRENT version and decide the action
    cur = (spark.table(DIM).filter("is_current = true")
           .select("customer_id",
                   F.col("city").alias("c_city"),
                   F.col("loyalty_tier").alias("c_tier"),
                   F.col("valid_from").alias("c_from")))
    j = (latest.join(cur, "customer_id", "left")
         .withColumn("action",
            F.when(F.col("c_from").isNotNull() & (F.col("event_ts") <= F.col("c_from")), "stale")
             .when((F.col("op") == "D") & F.col("c_from").isNull(), "reject_delete_unknown")
             .when(F.col("op") == "D", "close")
             .when(F.col("c_from").isNull(), "insert")
             .when(~F.col("city").eqNullSafe(F.col("c_city")) |
                   ~F.col("loyalty_tier").eqNullSafe(F.col("c_tier")), "version")
             .otherwise("same")))

    # save the decisions now: the MERGE below changes the dimension table
    (j.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(STAGE))
    stage = spark.table(STAGE)

    # 5) quarantine: invalid, stale and delete-for-unknown-customer events
    q = (bad
         .unionByName(stage.filter("action = 'stale'").select(*EV_COLS)
                      .withColumn("reject_reason", F.lit("stale event (older than current version)")))
         .unionByName(stage.filter("action = 'reject_delete_unknown'").select(*EV_COLS)
                      .withColumn("reject_reason", F.lit("delete for unknown customer")))
         .withColumn("processed_at", F.current_timestamp()))
    q.write.format("delta").mode("append").saveAsTable(QUAR)

    # 6) SCD Type 2: ONE atomic MERGE closes old versions and opens new ones
    cols = ["customer_id", "customer_name", "city", "loyalty_tier", "event_ts", "action"]
    closing = (stage.filter("action IN ('close', 'version')")
               .select(F.col("customer_id").alias("merge_key"), *cols))
    opening = (stage.filter("action IN ('insert', 'version')")
               .select(F.lit(None).cast("int").alias("merge_key"), *cols))
    closing.unionByName(opening).createOrReplaceTempView("cdc_staged")

    spark.sql(f"""
      MERGE INTO {DIM} AS t
      USING cdc_staged AS s
      ON t.customer_id = s.merge_key AND t.is_current = true
      WHEN MATCHED THEN UPDATE SET
        valid_to = s.event_ts, is_current = false, is_deleted = (s.action = 'close')
      WHEN NOT MATCHED AND s.merge_key IS NULL THEN INSERT
        (customer_sk, customer_id, customer_name, city, loyalty_tier,
         valid_from, valid_to, is_current, is_deleted)
        VALUES (xxhash64(s.customer_id, CAST(s.event_ts AS STRING)), s.customer_id,
                s.customer_name, s.city, s.loyalty_tier, s.event_ts,
                TIMESTAMP'9999-12-31 00:00:00', true, false)
    """)

    # 7) SCD Type 1: name corrections overwrite the name on ALL versions (no history)
    (stage.filter("action IN ('insert', 'version', 'same')")
          .select("customer_id", F.col("customer_name").alias("new_name"))
          .createOrReplaceTempView("cdc_names"))
    spark.sql(f"""
      MERGE INTO {DIM} AS t
      USING cdc_names AS s
      ON t.customer_id = s.customer_id
      WHEN MATCHED AND t.customer_name <> s.new_name THEN UPDATE SET customer_name = s.new_name
    """)

    # 8) log the batch
    c = {r["action"]: r["n"] for r in
         stage.groupBy("action").agg(F.count("*").alias("n")).collect()}
    row = [(batch_id, n_raw, n_bad, n_dupes, n_superseded,
            c.get("insert", 0), c.get("version", 0), c.get("close", 0), c.get("same", 0),
            c.get("stale", 0) + c.get("reject_delete_unknown", 0))]
    (spark.createDataFrame(row,
        "batch_id int, raw_events int, rejected_invalid int, duplicates_removed int, "
        "superseded_in_batch int, inserted int, new_versions int, closed int, "
        "no_change int, quarantined_other int")
     .withColumn("processed_at", F.current_timestamp())
     .write.format("delta").mode("append").saveAsTable(LOG))
    print(f"batch {batch_id}: done")

In [0]:
for b in (1, 2, 3):
    apply_batch(b)

display(spark.table(LOG).orderBy("batch_id"))

batch 1: already processed, skipping
batch 2: already processed, skipping
batch 3: already processed, skipping


batch_id,raw_events,rejected_invalid,duplicates_removed,superseded_in_batch,inserted,new_versions,closed,no_change,quarantined_other,processed_at
1,501,0,0,0,501,0,0,0,0,2026-10-28T17:15:49.769Z
2,165,3,15,0,0,72,3,72,0,2026-10-28T17:16:18.173Z
3,95,0,0,0,3,68,0,19,5,2026-10-28T17:16:40.684Z


In [0]:
d = spark.table(DIM)

# the customer with the most versions
cid = (d.groupBy("customer_id").count().orderBy(F.desc("count"), "customer_id").first()["customer_id"])
display(d.filter(F.col("customer_id") == cid).orderBy("valid_from"))

display(spark.table(QUAR).groupBy("reject_reason").count())
display(spark.table(QUAR).orderBy("processed_at").limit(20))

customer_sk,customer_id,customer_name,city,loyalty_tier,valid_from,valid_to,is_current,is_deleted
-3466376482129355152,1013,Nirja Pandya,Delhi,Plus,2026-09-01T08:00:00.000Z,2026-10-10T09:00:00.000Z,false,false
8159758573591463421,1013,Nirja Pandya,Chennai,Plus,2026-10-10T09:00:00.000Z,2026-10-20T09:00:00.000Z,false,false
-8013103098947230047,1013,Nirja Pandya,Mumbai,Plus,2026-10-20T09:00:00.000Z,9999-12-31T00:00:00.000Z,true,false


reject_reason,count
stale event (older than current version),5
missing customer_id,1
invalid op,1
missing attributes,1


event_id,batch_id,customer_id,op,customer_name,city,loyalty_tier,event_ts,reject_reason,processed_at
B2-bad-1,2,null,U,No Id,Mumbai,Basic,2026-10-10T09:00:00.000Z,missing customer_id,2026-10-28T17:16:01.451Z
B2-bad-2,2,1001,X,Bad Op,Mumbai,Basic,2026-10-10T09:00:00.000Z,invalid op,2026-10-28T17:16:01.451Z
B2-bad-3,2,1002,U,No City,null,Basic,2026-10-10T09:00:00.000Z,missing attributes,2026-10-28T17:16:01.451Z
B3-late-1283,3,1283,U,Kevin Palla,Chennai,Basic,2026-08-31T09:00:00.000Z,stale event (older than current version),2026-10-28T17:16:28.451Z
B3-late-1256,3,1256,U,Anvi Anand,Chennai,Basic,2026-08-31T09:00:00.000Z,stale event (older than current version),2026-10-28T17:16:28.451Z
B3-late-1345,3,1345,U,Eesha Solanki,Chennai,Basic,2026-08-31T09:00:00.000Z,stale event (older than current version),2026-10-28T17:16:28.451Z
B3-late-1327,3,1327,U,Anusha Bath,Chennai,Premium,2026-08-31T09:00:00.000Z,stale event (older than current version),2026-10-28T17:16:28.451Z
B3-late-1385,3,1385,U,Dipta Sarin,Chennai,Plus,2026-08-31T09:00:00.000Z,stale event (older than current version),2026-10-28T17:16:28.451Z


In [0]:
d = spark.table(DIM)
w2 = Window.partitionBy("customer_id").orderBy("valid_from")

checks = {
  "at most one current row per customer":
      d.filter("is_current = true").groupBy("customer_id").count().filter("count > 1").count(),
  "valid_from is before valid_to": d.filter("valid_from >= valid_to").count(),
  "no overlapping versions":
      d.withColumn("prev_to", F.lag("valid_to").over(w2))
       .filter("prev_to IS NOT NULL AND prev_to > valid_from").count(),
  "surrogate key is unique": d.count() - d.select("customer_sk").distinct().count(),
  "SCD1: one name per customer across all versions":
      d.groupBy("customer_id").agg(F.countDistinct("customer_name").alias("n")).filter("n > 1").count(),
  "every ordering customer exists in the dimension":
      spark.table(f"{CATALOG}.silver.orders").select("customer_id").distinct()
           .join(d.select("customer_id").distinct(), "customer_id", "left_anti").count(),
}
for name, bad_n in checks.items():
    print(("PASS " if bad_n == 0 else "FAIL ") + name)
assert all(v == 0 for v in checks.values()), "SCD2 checks failed"


PASS at most one current row per customer
PASS valid_from is before valid_to
PASS no overlapping versions
PASS surrogate key is unique
PASS SCD1: one name per customer across all versions
PASS every ordering customer exists in the dimension


In [0]:
fact = (spark.table(f"{CATALOG}.gold.fact_orders")
        .join(spark.table(f"{CATALOG}.gold.dim_date").select("date_key", "full_date"), "date_key"))
d = spark.table(DIM)

pit = (fact.join(d,
        (fact.customer_key == d.customer_id) &
        (fact.full_date >= F.to_date(d.valid_from)) &
        (fact.full_date <  F.to_date(d.valid_to)),
        "left"))

print("orders in fact:        ", fact.count())
print("orders after the join: ", pit.count())      # must be equal: no duplicated orders
print("orders with no version:", pit.filter("customer_sk IS NULL").count())

display(pit.groupBy("loyalty_tier")
           .agg(F.count("*").alias("orders"), F.sum("total_amount").alias("revenue"))
           .orderBy("loyalty_tier"))

orders in fact:         15500
orders after the join:  15500
orders with no version: 30


loyalty_tier,orders,revenue
null,30,170361.87
Basic,5007,35210628.72
Plus,5130,36701066.80
Premium,5333,37359046.11
